In [11]:
import pandas as pd
from sklearn.svm import SVC
from sklearn.datasets import load_iris
iris = load_iris()
df = pd.DataFrame(iris.data,columns = iris.feature_names)
df['flower'] = iris.target
df['flower'] = df["flower"].apply(lambda x : iris.target_names[x])
df.head()

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),flower
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


## By using traditional method of train test split we can get the score but it changes with every sample split

In [14]:
from sklearn.model_selection import train_test_split
Xtrain,Xtest,ytrain,ytest = train_test_split(iris.data,iris.target,test_size = 0.3)

In [15]:
model = SVC(kernel = "rbf",C = 30,gamma = "auto")
model.fit(Xtrain,ytrain)
model.score(Xtest,ytest)

0.9555555555555556

### Therefore we use K fold cross validation?

In [18]:
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_val_score

In [26]:
folds = StratifiedKFold()
cross_val_score(SVC(kernel = "linear",C = 10,gamma = "auto"),iris.data,iris.target,cv = 5)


array([1.        , 1.        , 0.9       , 0.96666667, 1.        ])

In [27]:
cross_val_score(SVC(kernel = "rbf",C = 10,gamma = "auto"),iris.data,iris.target,cv = 5)

array([0.96666667, 1.        , 0.96666667, 0.96666667, 1.        ])

In [28]:
cross_val_score(SVC(kernel = "rbf",C = 20,gamma = "auto"),iris.data,iris.target,cv = 5)

array([0.96666667, 1.        , 0.9       , 0.96666667, 1.        ])

### Above code we wrote is not so effecient to check weather which model is better for the given dataset

In [30]:
# Therefore we use loops

import numpy as np 
kernels = ['rbf','linear']
C = [1,10,20]
avg_scores = {}
for kval in kernels:
    for cval in C:
        cv_scores = cross_val_score(SVC(kernel = kval,C = cval,gamma = "auto"),iris.data,iris.target,cv = 5)
        avg_scores[kval + "_" + str(cval)] = np.average(cv_scores)
avg_scores

{'rbf_1': np.float64(0.9800000000000001),
 'rbf_10': np.float64(0.9800000000000001),
 'rbf_20': np.float64(0.9666666666666668),
 'linear_1': np.float64(0.9800000000000001),
 'linear_10': np.float64(0.9733333333333334),
 'linear_20': np.float64(0.9666666666666666)}

### This loop method is not optimal as it could take lots of iterations therefore sklearn has GridSearchCV a special library

In [31]:
from sklearn.model_selection import GridSearchCV

In [35]:
clf = GridSearchCV(SVC(gamma = "auto"),{
    'C' : [1,10,20],
    'kernel' : ["rbf","linear"]
},cv = 5,return_train_score = False)

clf.fit(iris.data,iris.target)
clf.cv_results_ # cv results are not easy to view therefore we will convert it into df

{'mean_fit_time': array([0.0009963 , 0.0008595 , 0.00088673, 0.0010694 , 0.00107217,
        0.0012537 ]),
 'std_fit_time': array([1.77321013e-04, 1.86420949e-04, 4.48663985e-05, 3.00282293e-04,
        4.41663323e-04, 3.17437398e-04]),
 'mean_score_time': array([0.00063996, 0.00060425, 0.00062456, 0.00077872, 0.000774  ,
        0.00160527]),
 'std_score_time': array([2.55539342e-05, 1.68317006e-04, 1.47408252e-05, 1.88119142e-04,
        3.94147451e-04, 9.37134656e-04]),
 'param_C': masked_array(data=[1, 1, 10, 10, 20, 20],
              mask=[False, False, False, False, False, False],
        fill_value=999999),
 'param_kernel': masked_array(data=['rbf', 'linear', 'rbf', 'linear', 'rbf', 'linear'],
              mask=[False, False, False, False, False, False],
        fill_value=np.str_('?'),
             dtype=object),
 'params': [{'C': 1, 'kernel': 'rbf'},
  {'C': 1, 'kernel': 'linear'},
  {'C': 10, 'kernel': 'rbf'},
  {'C': 10, 'kernel': 'linear'},
  {'C': 20, 'kernel': 'rbf'},
 

In [36]:
df = pd.DataFrame(clf.cv_results_)
df

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_C,param_kernel,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.000996,0.000177,0.000640,0.000026,1,rbf,"{'C': 1, 'kernel': 'rbf'}",0.966667,1.0,0.966667,0.966667,1.0,0.980000,0.016330,1
1,0.000859,0.000186,0.000604,0.000168,1,linear,"{'C': 1, 'kernel': 'linear'}",0.966667,1.0,0.966667,0.966667,1.0,0.980000,0.016330,1
2,0.000887,0.000045,0.000625,0.000015,10,rbf,"{'C': 10, 'kernel': 'rbf'}",0.966667,1.0,0.966667,0.966667,1.0,0.980000,0.016330,1
3,0.001069,0.000300,0.000779,0.000188,10,linear,"{'C': 10, 'kernel': 'linear'}",1.000000,1.0,0.900000,0.966667,1.0,0.973333,0.038873,4
4,0.001072,0.000442,0.000774,0.000394,20,rbf,"{'C': 20, 'kernel': 'rbf'}",0.966667,1.0,0.900000,0.966667,1.0,0.966667,0.036515,5
5,0.001254,0.000317,0.001605,0.000937,20,linear,"{'C': 20, 'kernel': 'linear'}",1.000000,1.0,0.900000,0.933333,1.0,0.966667,0.042164,6


In [37]:
# For best parameters
print(clf.best_params_)

{'C': 1, 'kernel': 'rbf'}


In [42]:
# we can also write it as 
params = {'C':[1,10,20] ,'kernel':['rbf',"linear"]}
grid = GridSearchCV(SVC(), params, cv=5)
grid.fit(iris.data,iris.target)

print(grid.best_params_)

{'C': 1, 'kernel': 'linear'}


### The problem with above technique of GridSearchCV is it tries all the combination therefore for large datasets and many parameters it is not optimal

#### Grid Search → tries all combinations

#### Therefore we can also use Random Search → tries random combinations

In [44]:
from sklearn.model_selection import RandomizedSearchCV
rs = RandomizedSearchCV(SVC(gamma = "auto"),{
            'C' : [1,10,20],
            'kernel' : ['rbf','linear']
},
cv = 5,
return_train_score = False,n_iter = 6)
rs.fit(iris.data,iris.target)

,estimator,SVC(gamma='auto')
,param_distributions,"{'C': [1, 10, ...], 'kernel': ['rbf', 'linear']}"
,n_iter,6
,scoring,None
,n_jobs,None
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,random_state,None
,error_score,nan


In [47]:
rs.best_params_

{'kernel': 'rbf', 'C': 1}

### Now lets find the best model using hyper parameter tuning we just did above

In [48]:
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

In [53]:
model_params = {
    
'svm' : {
'model' : SVC(gamma = "auto"),
'params' : {
'C' : [1,10,20],
'kernel' : ["rbf", "linear"]
}
},
    
'random_forest' : {
'model' : RandomForestClassifier(),
'params' : {
'n_estimators' : [1,5,10]
}
},
    
'logistic_regression' : {
'model' : LogisticRegression(max_iter=1000),
'params' : {
'C' : [1,5,10]
}
}
    
}

In [55]:
scores = []
for model_name,mp in model_params.items():
    clf = GridSearchCV(mp['model'],mp['params'],cv = 5 , return_train_score = False)
    clf.fit(iris.data,iris.target)
    scores.append({
        "model" : model_name,
        "best_score" : clf.best_score_,
        "best_params" : clf.best_params_
    })

In [57]:
df = pd.DataFrame(scores,columns = ['model','best_score','best_params'])
df

,model,best_score,best_params
0,svm,0.980000,"{'C': 1, 'kernel': 'rbf'}"
1,random_forest,0.960000,{'n_estimators': 5}
2,logistic_regression,0.973333,{'C': 1}
